# Notebook 05: CUPED

## Research question

Do the available pre-treatment covariates provide enough predictive information to improve the precision of the treatment-effect estimates?

## Analytical objective

Measure the predictive value of candidate covariates, implement single- and multiple-covariate CUPED adjustments, and compare variance reduction and inference before and after adjustment.

## Statistical framework

For a covariate X and outcome Y, use `Y_cuped = Y - theta(X - mean(X))`, with `theta = Cov(Y, X) / Var(X)`. Under randomization, this adjustment targets precision rather than confounding control. The estimand in this notebook is the intention-to-treat mean difference, `Mens E-Mail - No E-Mail`.

## Assumptions and limitations

CUPED requires covariates measured before random assignment, covariate balance in expectation, and a stable predictive relationship with the outcome. The adjustment preserves the outcome's overall mean, but a realized treatment-effect estimate may move slightly because the experiment has finite-sample covariate imbalance. Coefficients are estimated on the same experiment for this tutorial; in smaller or highly adaptive experiments, pre-period estimation or cross-fitting can reduce overfitting concerns.

## Required output

Rank covariates by R², report variance reduction and equivalent sample savings, compare single- and multi-covariate adjustment, and explain the inferential consequences for binary outcomes.

## Why CUPED works

The unadjusted treatment-effect estimator is the difference in outcome means, $\hat\tau_{raw}=\bar Y_T-\bar Y_C$. CUPED removes the part of the outcome that is linearly predictable from a pre-treatment covariate $X$:

$$Y_{cuped}=Y-\theta(X-\bar X).$$

The corresponding treatment-effect estimate is

$$\hat\tau_{cuped}=\hat\tau_{raw}-\theta(\bar X_T-\bar X_C).$$

Random assignment implies $E[\bar X_T-\bar X_C]=0$, so the adjustment preserves the expected intention-to-treat effect. In a finite sample the covariate means will not be exactly equal, which is why the adjusted point estimate can move slightly. This is precision adjustment, not a change in the causal estimand.

For one covariate, the adjusted outcome variance is

$$Var(Y_{cuped})=Var(Y)+\theta^2Var(X)-2\theta Cov(Y,X).$$

Minimizing this expression gives $\theta^*=Cov(Y,X)/Var(X)$. At that value,

$$Var(Y_{cuped})=Var(Y)(1-\rho_{Y,X}^2).$$

Therefore the theoretical variance reduction is $\rho^2$, which equals the $R^2$ from a simple OLS regression with an intercept. A predictive pre-period variable produces a useful gain; a variable with near-zero $R^2$ cannot. With multiple covariates, OLS estimates a coefficient vector and the same idea becomes regression residualization, with variance reduction close to the model $R^2$.

### How the implementation follows the theory

1. Restrict the sample to `Mens E-Mail` and `No E-Mail`, matching the estimand.
2. Regress each outcome on every candidate pre-treatment covariate and rank predictive $R^2$.
3. For single-covariate CUPED, estimate `theta`, center the covariate, and subtract its predicted component from the outcome.
4. For multi-covariate CUPED, fit `outcome ~ history + recency`, center the fitted values, and subtract them.
5. Verify that the overall outcome mean is preserved, quantify variance and sample-size savings, and compare treatment arms with the same Welch mean test before and after adjustment.

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
import statsmodels.formula.api as smf


In [2]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

hillstrom_df.info()

display(hillstrom_df.head(), hillstrom_df.tail())

# Keep the analysis population aligned with the downstream estimand.
analysis_df = hillstrom_df[
    hillstrom_df["segment"].isin(["Mens E-Mail", "No E-Mail"])
].copy()
analysis_df["treatment"] = (
    analysis_df["segment"] == "Mens E-Mail"
).astype(int)

print(f"Analysis sample: {len(analysis_df):,} customers")
display(analysis_df.groupby("segment").size().rename("n"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


Analysis sample: 42,613 customers


segment
Mens E-Mail    21307
No E-Mail      21306
Name: n, dtype: int64

### Step 1: select covariates using predictive R²

For each candidate covariate and outcome, the code fits a separate OLS model and records R². `C(...)` tells statsmodels to dummy-code categorical variables rather than treat their labels as ordered numbers. Higher R² means more predictable outcome variation and more room for CUPED to reduce noise. Covariates must be measured before treatment; adjusting for a post-treatment variable could remove part of the treatment effect and introduce bias. Raw R² also tends to favor categorical predictors with more levels, so these rankings are descriptive and should ideally be confirmed on historical or held-out data.

In [3]:
candidate_covariates = {
    "recency": "recency",
    "history_segment": "C(history_segment)",
    "history": "history",
    "mens": "mens",
    "womens": "womens",
    "zip_code": "C(zip_code)",
    "newbie": "newbie",
    "channel": "C(channel)",
}
outcomes = ["visit", "conversion", "spend"]

r2_rows = []
for outcome in outcomes:
    for covariate, formula_term in candidate_covariates.items():
        model = smf.ols(
            formula=f"{outcome} ~ {formula_term}",
            data=analysis_df,
        ).fit()
        r2_rows.append({
            "outcome": outcome,
            "covariate": covariate,
            "r_squared": model.rsquared,
        })

r2_df = (
    pd.DataFrame(r2_rows)
    .sort_values(["outcome", "r_squared"], ascending=[True, False])
    .reset_index(drop=True)
)
r2_df["rank_within_outcome"] = (
    r2_df.groupby("outcome")["r_squared"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(r2_df.round(6))
display(
    r2_df.pivot(index="covariate", columns="outcome", values="r_squared")
    .sort_values("spend", ascending=False)
    .round(6)
)

,outcome,covariate,r_squared,rank_within_outcome
0,conversion,history_segment,0.001307,1
1,conversion,history,0.000888,2
2,conversion,recency,0.000661,3
3,conversion,newbie,0.000319,4
4,conversion,channel,0.000164,5
5,conversion,mens,0.000111,6
6,conversion,zip_code,0.000068,7
7,conversion,womens,0.000037,8
8,spend,history_segment,0.000752,1
9,spend,history,0.000438,2


outcome,conversion,spend,visit
covariate,,,
history_segment,0.001307,0.000752,0.006840
history,0.000888,0.000438,0.004444
recency,0.000661,0.000272,0.006414
mens,0.000111,0.000205,0.001157
newbie,0.000319,0.000154,0.006390
channel,0.000164,0.000049,0.002546
zip_code,0.000068,0.000021,0.003068
womens,0.000037,0.000000,0.000692


All candidate variables have very low predictive power. `history_segment` has the largest in-sample individual R² for each outcome, but it is a discretized version of `history` and uses several fitted parameters. The following analysis uses the two continuous, nonredundant measures `history` and `recency`, which directly represent prior purchasing intensity and timing. Because their R² values are small, only modest variance reduction should be expected. In a production experiment, covariates and transformations should preferably be chosen from prior data rather than selected after viewing the experiment outcomes.

### Step 2: apply single-covariate CUPED

The function below implements `Y_cuped = Y - theta * (X - mean(X))`, where `theta = Cov(Y, X) / Var(X)`. Centering $X$ is essential: it makes the subtracted term average to zero and therefore preserves the overall outcome mean. The code creates one adjusted column for every outcome-covariate pair, checks mean preservation, and reports both percentage variance reduction and the equivalent number of users saved at the current sample size.

In [4]:
def cuped_adjustment(df, outcome, covariate):
    """Return the centered CUPED outcome and its estimated theta."""
    covariance = df[[outcome, covariate]].cov().loc[outcome, covariate]
    covariate_variance = df[covariate].var(ddof=1)
    if np.isclose(covariate_variance, 0):
        raise ValueError(f"{covariate} has zero variance")

    theta = covariance / covariate_variance
    adjusted = df[outcome] - theta * (
        df[covariate] - df[covariate].mean()
    )
    return adjusted, theta

single_cuped_rows = []
for outcome in outcomes:
    raw_variance = analysis_df[outcome].var(ddof=1)

    for covariate in ["history", "recency"]:
        column = f"{outcome}_cuped_{covariate}"
        analysis_df[column], theta = cuped_adjustment(
            analysis_df, outcome, covariate
        )
        adjusted_variance = analysis_df[column].var(ddof=1)
        variance_reduction = 1 - adjusted_variance / raw_variance

        single_cuped_rows.append({
            "outcome": outcome,
            "covariate": covariate,
            "theta": theta,
            "raw_variance": raw_variance,
            "adjusted_variance": adjusted_variance,
            "variance_reduction_pct": 100 * variance_reduction,
            "sample_size_saving_pct": 100 * variance_reduction,
            "equivalent_users_saved": len(analysis_df) * variance_reduction,
            "mean_preserved": np.isclose(
                analysis_df[column].mean(),
                analysis_df[outcome].mean(),
            ),
        })

single_cuped_df = pd.DataFrame(single_cuped_rows)
display(single_cuped_df.round(6))

,outcome,covariate,theta,raw_variance,adjusted_variance,variance_reduction_pct,sample_size_saving_pct,equivalent_users_saved,mean_preserved
0,visit,history,0.000091,0.123596,0.123047,0.444388,0.444388,189.367261,True
1,visit,recency,-0.008032,0.123596,0.122804,0.641438,0.641438,273.335774,True
2,conversion,history,0.000011,0.009046,0.009038,0.088779,0.088779,37.831588,True
3,conversion,recency,-0.000698,0.009046,0.009040,0.066114,0.066114,28.173220,True
4,spend,history,0.001223,224.894096,224.795648,0.043775,0.043775,18.653967,True
5,spend,recency,-0.070503,224.894096,224.833017,0.027159,0.027159,11.573347,True


After CUPED, adjusted `visit` and `conversion` values are continuous and may even fall slightly outside $[0,1]$. That is acceptable because they are variance-reduced analysis variables, not predicted probabilities, but a proportion z-test is no longer appropriate. A Welch t-test on the adjusted mean or a regression with robust standard errors can be used instead. With these large arms, a Welch test on the raw 0/1 outcome is asymptotically equivalent to an unpooled test of proportions, which makes the raw-versus-adjusted precision comparison coherent. Variance reduction and equivalent sample-size savings remain the clearest measures of gain.

### Compare the original and CUPED-adjusted spend between the two segments using t-tests

In [5]:
def welch_comparison(df, outcome_column, method):
    treated = df.loc[df["segment"] == "Mens E-Mail", outcome_column]
    control = df.loc[df["segment"] == "No E-Mail", outcome_column]
    statistic, p_value = ttest_ind(treated, control, equal_var=False)
    difference = treated.mean() - control.mean()
    standard_error = np.sqrt(
        treated.var(ddof=1) / len(treated)
        + control.var(ddof=1) / len(control)
    )

    return {
        "method": method,
        "difference": difference,
        "standard_error": standard_error,
        "ci_low": difference - 1.96 * standard_error,
        "ci_high": difference + 1.96 * standard_error,
        "t_statistic": statistic,
        "p_value": p_value,
    }

single_spend_inference_df = pd.DataFrame([
    welch_comparison(analysis_df, "spend", "Raw spend"),
    welch_comparison(
        analysis_df, "spend_cuped_history", "CUPED: history"
    ),
    welch_comparison(
        analysis_df, "spend_cuped_recency", "CUPED: recency"
    ),
])
single_spend_display_df = single_spend_inference_df.copy()
single_spend_display_df["p_value"] = (
    single_spend_display_df["p_value"].map(lambda value: f"{value:.3e}")
)
display(single_spend_display_df.round(6))

,method,difference,standard_error,ci_low,ci_high,t_statistic,p_value
0,Raw spend,0.769827,0.145247,0.485144,1.05451,5.300140,1.164e-07
1,CUPED: history,0.767438,0.145215,0.482817,1.05206,5.284842,1.265e-07
2,CUPED: recency,0.771516,0.145227,0.486871,1.05616,5.312494,1.088e-07


### Step 3: combine covariates with regression adjustment

The regression version estimates both coefficients jointly with `outcome ~ history + recency`. The code centers the fitted values before subtracting them, so the adjusted outcome retains the original mean. Algebraically, this is multivariate CUPED: OLS chooses the coefficient vector that minimizes residual variance. The reported variance reduction should equal the model R² up to numerical precision. Because `history` and `recency` overlap in what they predict, the joint gain need not equal the sum of their separate gains.

In [6]:
multi_cuped_rows = []
multi_models = {}

for outcome in outcomes:
    model = smf.ols(
        formula=f"{outcome} ~ history + recency",
        data=analysis_df,
    ).fit()
    multi_models[outcome] = model

    # Center fitted values so the adjusted outcome keeps its original mean.
    centered_prediction = model.fittedvalues - model.fittedvalues.mean()
    column = f"{outcome}_cuped_multi"
    analysis_df[column] = analysis_df[outcome] - centered_prediction

    raw_variance = analysis_df[outcome].var(ddof=1)
    adjusted_variance = analysis_df[column].var(ddof=1)
    variance_reduction = 1 - adjusted_variance / raw_variance

    multi_cuped_rows.append({
        "outcome": outcome,
        "model_r_squared": model.rsquared,
        "history_coefficient": model.params["history"],
        "recency_coefficient": model.params["recency"],
        "raw_variance": raw_variance,
        "adjusted_variance": adjusted_variance,
        "variance_reduction_pct": 100 * variance_reduction,
        "sample_size_saving_pct": 100 * variance_reduction,
        "equivalent_users_saved": len(analysis_df) * variance_reduction,
        "mean_preserved": np.isclose(
            analysis_df[column].mean(),
            analysis_df[outcome].mean(),
        ),
    })

multi_cuped_df = pd.DataFrame(multi_cuped_rows)
display(multi_cuped_df.round(6))

,outcome,model_r_squared,history_coefficient,recency_coefficient,raw_variance,adjusted_variance,variance_reduction_pct,sample_size_saving_pct,equivalent_users_saved,mean_preserved
0,visit,0.008778,0.000069,-0.006808,0.123596,0.122511,0.877801,0.877801,374.057128,True
1,conversion,0.001250,0.000009,-0.000532,0.009046,0.009034,0.124984,0.124984,53.259598,True
2,spend,0.000576,0.001051,-0.051771,224.894096,224.764668,0.057551,0.057551,24.524099,True


### Compare the original and CUPED multiple covariate adjusted spend between the two segments using t-tests

In [7]:
multi_spend_inference_df = pd.DataFrame([
    welch_comparison(analysis_df, "spend", "Raw spend"),
    welch_comparison(
        analysis_df,
        "spend_cuped_multi",
        "CUPED: history + recency",
    ),
])
multi_spend_display_df = multi_spend_inference_df.copy()
multi_spend_display_df["p_value"] = (
    multi_spend_display_df["p_value"].map(lambda value: f"{value:.3e}")
)
display(multi_spend_display_df.round(6))

,method,difference,standard_error,ci_low,ci_high,t_statistic,p_value
0,Raw spend,0.769827,0.145247,0.485144,1.054510,5.300140,1.164e-07
1,CUPED: history + recency,0.769015,0.145205,0.484413,1.053616,5.296069,1.190e-07


### Compare raw and multi-covariate inference for every outcome

Using the same Welch mean test on both versions isolates the practical effect of adjustment. For the raw binary outcomes this is a test of a difference in means (proportions); after CUPED it remains a test of the adjusted mean difference, even though individual adjusted values are no longer binary.

In [8]:
all_outcome_inference_rows = []
for outcome in outcomes:
    for outcome_column, method in [
        (outcome, "Raw"),
        (f"{outcome}_cuped_multi", "CUPED: history + recency"),
    ]:
        result = welch_comparison(analysis_df, outcome_column, method)
        result["outcome"] = outcome
        all_outcome_inference_rows.append(result)

all_outcome_inference_df = pd.DataFrame(all_outcome_inference_rows)[[
    "outcome",
    "method",
    "difference",
    "standard_error",
    "ci_low",
    "ci_high",
    "t_statistic",
    "p_value",
]]
all_outcome_display_df = all_outcome_inference_df.copy()
all_outcome_display_df["p_value"] = (
    all_outcome_display_df["p_value"].map(lambda value: f"{value:.3e}")
)
display(all_outcome_display_df.round(6))

,outcome,method,difference,standard_error,ci_low,ci_high,t_statistic,p_value
0,visit,Raw,0.076590,0.003386,0.069953,0.083226,22.620231,1.364e-112
1,visit,CUPED: history + recency,0.076618,0.003371,0.070012,0.083225,22.730045,1.161e-113
2,conversion,Raw,0.006805,0.000921,0.005000,0.008610,7.389736,1.502e-13
3,conversion,CUPED: history + recency,0.006800,0.000920,0.004996,0.008603,7.388525,1.516e-13
4,spend,Raw,0.769827,0.145247,0.485144,1.054510,5.300140,1.164e-07
5,spend,CUPED: history + recency,0.769015,0.145205,0.484413,1.053616,5.296069,1.190e-07


## CUPED takeaway

### What the results say

The available pre-treatment covariates are weak predictors of the two-week outcomes. The largest single-predictor R² values are approximately 0.00684 for `visit`, 0.00131 for `conversion`, and 0.00075 for `spend`, all from `history_segment`. For the selected continuous covariates, `history` reduces variance by about 0.444%, 0.089%, and 0.044% for visit, conversion, and spend; `recency` reduces it by about 0.641%, 0.066%, and 0.027%. These values match the corresponding simple-regression R² values, as the CUPED variance identity predicts.

Combining `history` and `recency` improves the reduction to about 0.878% for visits, 0.125% for conversions, and 0.058% for spend. At the current sample size of 42,613, those gains are equivalent to saving only about 374, 53, and 25 users respectively while retaining the original precision. CUPED works mathematically here, but the input covariates simply contain too little information about the future outcomes to matter operationally.

### Effect on inference

For spend, the raw estimate is about $0.770 per customer with a 95% confidence interval of [$0.485, $1.055] and a Welch t-statistic of 5.300. The history-, recency-, and multi-covariate t-statistics are 5.285, 5.312, and 5.296. The adjusted estimates move by only a few tenths of a cent and all p-values remain around $10^{-7}$. A variance-reduction method does not guarantee a larger t-statistic in one realized experiment: the estimate as well as its standard error changes when chance covariate imbalance is removed. The relevant question is whether repeated experiments would have lower estimator variance; here the answer is yes, but only trivially.

### Practical recommendation

Do not rely on the available `history` and `recency` fields for meaningful sample-size reduction. CUPED would be more useful with a pre-experiment measurement closely aligned to each outcome, such as visits, conversions, and spend during a comparable earlier two-week window. Several repeated pre-period measurements, email engagement, and recent category-level purchasing could further improve prediction. Those features must be defined before randomization and checked for coverage, missingness, temporal stability, and treatment independence. CUPED also does not repair non-random treatment assignment; that requires a causal identification strategy rather than variance reduction.